This is the Spark version

In [ ]:
#Java Environment Setup
import os
import sys
import glob

venv_path = sys.prefix
jvm_dir = os.path.join(venv_path, "jvm")
java_execs = glob.glob(os.path.join(jvm_dir, "**/bin/java"), recursive=True)

if java_execs:
    resolved_java_home = os.path.dirname(os.path.dirname(os.path.abspath(java_execs[0])))
    os.environ["JAVA_HOME"] = resolved_java_home
    os.environ["PATH"] = os.path.join(resolved_java_home, "bin") + os.pathsep + os.environ.get("PATH", "")
    print(f"JAVA_HOME configured at: {resolved_java_home}")
else:
    print("Warning: JDK binary not found in .venv/jvm")

In [ ]:
import sys
import json
import math
import sqlite3
from datetime import datetime
from kafka import KafkaConsumer
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="pyspark")
with open("config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

BOOTSTRAP_SERVER = config["kafka"]["bootstrap_server"]
TOKENS_TOPIC = config["kafka"]["tokens_topic"]
BASELINES_DB = config["baselines_db_path"]

#load baselines into memory for fast lookups
print("attempritng to connect to db")
conn = sqlite3.connect(BASELINES_DB)
cursor = conn.cursor()
cursor.execute("SELECT * FROM token_baselines")
#currently loads all token baselines for all hours. can be configured
baseline_lookup = {row[0]: list(row[1:]) for row in cursor.fetchall()}
conn.close()

print(f"Loaded baselines for {len(baseline_lookup)} tokens.")

#init spark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkTokensCounter") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark session initialized.")

In [ ]:
#this function evaluates a token for 2 diffrent levels
#trend: a rise in apperenses, but over time. this signals a rise in popularity for example
#spike: a mommentery rise, very fast, this signals an event.

#all calculations scale with baseline, and have a floor for rare words
#the function returns a packet of informations that can be sent to diffrent notifications functions

import pandas as pd
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType, BooleanType
)

# insted of  bucket_state  we had before, no we have a spark struct
state_schema = StructType([
    StructField("spike_level", DoubleType()),
    StructField("trend_level", DoubleType()),
    StructField("is_spike", BooleanType()),
    StructField("is_trend", BooleanType()),
    StructField("last_ts", LongType())
])

# define the output, we return a lot of details for debug utility, but a simple token and alert will do
output_schema = StructType([
    StructField("token", StringType()),
    StructField("ts", LongType()),
    StructField("alert_type", StringType()),
    StructField("spike_level", DoubleType()),
    StructField("trend_level", DoubleType()),
    StructField("thresh_spike", DoubleType()),
    StructField("thresh_trend", DoubleType()),
    StructField("baseline_hourly", DoubleType())
])

#rewriten to utilise spark in micro batches, insted of one by one using python. better for bigger data
def evaluate_token(key, pdf: pd.DataFrame, state):
    #create df in needed
    if not isinstance(pdf, pd.DataFrame):
        dfs = list(pdf)
        if not dfs:
            return pd.DataFrame([], columns=[
                "token", "ts", "alert_type", "spike_level", "trend_level",
                "thresh_spike", "thresh_trend", "baseline_hourly"
            ])
        pdf = pd.concat(dfs, ignore_index=True)

    if pdf.empty:
        return pd.DataFrame([], columns=[
            "token", "ts", "alert_type", "spike_level", "trend_level",
            "thresh_spike", "thresh_trend", "baseline_hourly"
        ])
    
    token = key[0]
    alerts = []

    if state.exists:
        spike_level, trend_level, is_spike, is_trend, last_ts = state.get
    else:
        #init
        spike_level, trend_level, is_spike, is_trend = 0.0, 0.0, False, False
        last_ts = int(pdf["ts"].iloc[0])

    #sort microbatch by arrivle time. the same token must be evaluated in order of arrivle.
    pdf = pdf.sort_values("ts")

    for index, row in pdf.iterrows():
        ts = int(row["ts"])
        delta_t = max(0.0, float(ts - last_ts))

        #find time of day to select the proper baseline
        hour = datetime.fromtimestamp(ts).hour
        baseline_hourly = baseline_lookup.get(token, [0.01] * 24)[hour]

        #calculate drain for the 2 levels. spike drains fast and trend slower
        #added a floor to drain rate, to handle rare words
        drain_spike = max(6.0 / 3600.0, 4.0 * (baseline_hourly / 3600.0)) * delta_t
        drain_trend = max(0.5 / 3600.0, 1.2 * (baseline_hourly / 3600.0)) * delta_t

        #calculate the new levels
        spike_level = max(0.0, spike_level - drain_spike + 1.0)
        trend_level = max(0.0, trend_level - drain_trend + 1.0)

        #thresh holds have floors to avoid rare words getting to many spikes
        #spike thresh hold is lower than trend
        baseline_sqrt = math.sqrt(baseline_hourly)
        thresh_spike = max(7.0, 3.0 * baseline_sqrt)
        thresh_trend = max(15.0, 5.0 * baseline_sqrt)

        if not is_spike and spike_level >= thresh_spike:
            is_spike = True
            alerts.append((token, ts, "NOTIFY SPIKE", spike_level, trend_level, thresh_spike, thresh_trend, baseline_hourly))
        elif is_spike and spike_level < (thresh_spike * 0.75):
            is_spike = False

        if not is_trend and trend_level >= thresh_trend:
            is_trend = True
            alerts.append((token, ts, "NOTIFY TREND", spike_level, trend_level, thresh_spike, thresh_trend, baseline_hourly))
        elif is_trend and trend_level < (thresh_trend * 0.75):
            is_trend = False

        last_ts = ts

    #update state value to match
    state.update((spike_level, trend_level, is_spike, is_trend, last_ts))

    return [pd.DataFrame(alerts, columns=[
        "token", "ts", "alert_type", "spike_level", "trend_level",
        "thresh_spike", "thresh_trend", "baseline_hourly"
    ])]

In [ ]:
#recives results from evaluate token and prints to screen. can be changed to any channle like telgram, notifications and so on
def notify_user(eval_res):
    time_str = datetime.fromtimestamp(eval_res["ts"]).strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{time_str}] {eval_res['alert_type']}: '{eval_res['token']}'")

def notify_user_debug(eval_res):
    time_str = datetime.fromtimestamp(eval_res["ts"]).strftime("%Y-%m-%d %H:%M:%S")
    if eval_res["alert_type"] == "NOTIFY SPIKE":
        print(f"[{time_str}] NOTIFY SPIKE: Token: '{eval_res['token']}' | Level: {eval_res['spike_level']:.2f} / {eval_res['thresh_spike']:.1f} | Base: {eval_res['baseline_hourly']:.3f}/h")
    elif eval_res["alert_type"] == "NOTIFY TREND":
        print(f"[{time_str}] NOTIFY TREND: Token: '{eval_res['token']}' | Level: {eval_res['trend_level']:.2f} / {eval_res['thresh_trend']:.1f} | Base: {eval_res['baseline_hourly']:.3f}/h")

In [ ]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, LongType


raw_stream_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVER) \
    .option("subscribe", TOKENS_TOPIC) \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

val_schema = StructType([StructField("ts", LongType(), True)])

parsed_tokens_df = raw_stream_df.select(
    col("key").cast("string").alias("token"),
    from_json(col("value").cast("string"), val_schema).alias("data")
).select("token", "data.ts").filter(col("token").isNotNull() & col("ts").isNotNull())

alerts_stream_df = parsed_tokens_df \
    .groupBy("token") \
    .applyInPandasWithState(
        func=evaluate_token,
        outputStructType=output_schema,
        stateStructType=state_schema,
        outputMode="append",
        timeoutConf="NoTimeout"
    )

def display_alerts(batch_df, batch_id):
    for row in batch_df.collect():
        notify_user_debug(row.asDict())
print("Stage 2 worker live: Spark version")

#token levels are kept at tmp/spark-stateful-tokens-checkpoint, can be changed
query = alerts_stream_df.writeStream \
    .foreachBatch(display_alerts) \
    .option("checkpointLocation", "/tmp/spark-stateful-tokens-checkpoint") \
    .outputMode("append") \
    .start()

try:
    query.awaitTermination()
except KeyboardInterrupt:
    query.stop()
    print("\nSpark streaming worker stopped.")

In [ ]:
#rm -rf /tmp/spark-stateful-tokens-checkpoint